### Concept 1: @property

In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance

    @property
    def balance(self):
        return self._balance

acc1 = Account("Kavya", 100)

print("acc1.balance:", acc1.balance)

acc1.balance = 500

#### @property isn't "a keyword meaning getter" — it's decorator syntax applying an ordinary built-in class, property, whose specific job happens to be "let this method be accessed with attribute syntax."

##### any method turned into a @property getter can only ever take self — nothing else — because obj.value syntax provides no mechanism to pass anything in. That's precisely why @property is meant for values, not for functions that need custom input each time you call them — if a method genuinely needs an argument to do its job, it has to stay a normal method (obj.method(arg)), not a property.

In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            print("  Rejected: balance cannot be negative")
            return
        self._balance = value

prop = Account.__dict__['balance']
print("prop:", prop)
print("type(prop):", type(prop))
print("prop.fget:", prop.fget)
print("prop.fset:", prop.fset)

#### @property above def balance(self): → this defines the getter.
#### @balance.setter above def balance(self, value): → this defines the setter. The rule you noticed is correct: the method name must match (balance both times), and the decorator is written as @<that name>.setter.
#### The result: Account.__dict__['balance'] holds one single property-type object, and that one object knows about both your getter and your setter.

#### you're right that a plain method can do everything a getter/setter needs to do. The actual reason @property matters isn't "you couldn't validate otherwise" — it's that switching a plain attribute to validated get/set methods silently breaks (or worse, silently corrupts) any existing code that used the old obj.attribute syntax, while switching to @property doesn't, because the calling syntax never has to change. That's the specific, real-world engineering reason it exists — not raw capability, but safe evolution of a class without rewriting every caller.

#### Example 1: Validation

In [1]:
class ModelConfig:
    def __init__(self, temperature):
        self.temperature = temperature

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        if not (0 <= value <= 1):
            raise ValueError(f"temperature must be between 0 and 1, got {value}")
        self._temperature = value

config = ModelConfig(0.7)
print(config.__dict__)
config.temperature = 0.5
print(ModelConfig.__dict__)

{'_temperature': 0.7}
{'__module__': '__main__', '__firstlineno__': 1, '__init__': <function ModelConfig.__init__ at 0x10b18e980>, 'temperature': <property object at 0x10b1a8a90>, '__static_attributes__': ('_temperature', 'temperature'), '__dict__': <attribute '__dict__' of 'ModelConfig' objects>, '__weakref__': <attribute '__weakref__' of 'ModelConfig' objects>, '__doc__': None}


#### Example 2: Computed value, nothing stored at all

In [2]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height   # never stored - computed fresh every time

r = Rectangle(4, 5)
r.width = 10
print(r.area)

50


#### Example 3: Side effect on access (not just validation)

In [3]:
class Account:
    @property
    def balance(self):
        print(f"  [AUDIT] {self.owner}'s balance was read")
        return self._balance

    @balance.setter
    def balance(self, value):
        print(f"  [AUDIT] {self.owner}'s balance changed: {self._balance} -> {value}")
        self._balance = value

### Concept 3: @x.deleter

In [4]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        self._balance = value

    @balance.deleter
    def balance(self):
        print(f"  Deleting {self.owner}'s balance record")
        del self._balance

acc1 = Account("Kavya", 100)
print("acc1.balance:", acc1.balance)

del acc1.balance

print(acc1.balance)

acc1.balance: 100
  Deleting Kavya's balance record


AttributeError: 'Account' object has no attribute '_balance'

del acc1.balance normally would just remove a key from acc1.__dict__ directly. @balance.deleter lets you intercept that too — run your own code (logging, cleanup, whatever) before actually deleting _balance. After it runs, _balance is genuinely gone — trying to read acc1.balance afterward fails, since the getter has nothing left to return.

The deleter completes the action by deleting the key from the instance dictionary (self.__dict__), not the data descriptor

If you just do acc1.balance = -500, Python by default looks at the instance, says "Okay, I'll store -500 under the key 'balance'" and walks away. It doesn't care if the number makes sense.You use @property strictly to intercept that action.Think of @property as a trapdoor built into the class. When you type acc1.balance = -500:Python starts to look at the instance.But wait! Python notices the class has a data descriptor named balance.Because it's a data descriptor, Python diverts the assignment away from the instance dictionary and hands it to the setter function instead.The setter function runs your safety check (if value < 0).Only after the safety check passes does the setter manually write the data into the instance under a different, hidden name (self._balance).

"Can't I just call acc1.balance directly?"Yes, you absolutely can! And if you don't care about safety, typos, or validation, you should just use acc1.balance without any properties. Python developers prefer plain attributes for simple data.You only add @property when you need that "interceptor" to:Validate: Check if a balance is negative or an email is missing an @ symbol.Protect: Make a variable read-only (by omitting the setter entirely).Calculate: Compute a value on the fly (like calculating age from a birthdate).